# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an interactive walk-through for exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed (uncomment in Colab or a clean environment)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
dataset = mlc.Dataset(croissant_url)

# Access metadata fields from the dataset
print(f"Dataset name: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.date_published}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets and their field `@id`s for data exploration.

Below, we enumerate available Record Sets, fields, and their IDs using `mlcroissant`'s metadata structure.

In [ ]:
# List all available record sets and their associated fields, using their @id fields
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record set name: {rs.name} (id: {rs.id})")
    print("  Fields:")
    for fld in rs.fields:
        print(f"    - {fld.name} (id: {fld.id}, data type: {fld.data_type})")
    print()

Let's inspect a sample of records from one of the record sets. (Change the variable below to explore other record sets.)

In [ ]:
# Select a record set by its @id (from above e.g., record_sets[0].id)
example_record_set_id = record_sets[0].id  # use the first record set for the demo
print(f"Sample records from record set: {example_record_set_id}\n")
for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
    print(record)
    if i >= 2:  # show 3 records
        break

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis. We use the `@id` of each record set for clarity.

In [ ]:
# Collect and load all record sets' data into Pandas DataFrames (referenced by @id)
rs_ids = [rs.id for rs in record_sets]
dataframes = {}

for rs_id in rs_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Record set '{rs_id}': Shape {df.shape}")

# Show columns (field @id's) for the first record set
print(f"\nColumns for first record set ({rs_ids[0]}):\n", dataframes[rs_ids[0]].columns.tolist())
dataframes[rs_ids[0]].head()

## 4. Exploratory Data Analysis (EDA)
Let's demonstrate filtering, normalization, and grouping using numeric fields and groupings referenced by their `@id` fields. Modify the `numeric_field_id` and `group_field_id` as needed for your analysis.

In [ ]:
# --- CHOOSE fields by @id below (refer to Data Overview step) ---
record_set_id = rs_ids[0]  # usually the primary/protein-level record set
df = dataframes[record_set_id]

# Example: assuming a field representing MW (molecular weight) exists, or pI, or coverage
# Find an appropriate numeric and group field by checking field names/ids above
numeric_field_id = None
group_field_id = None
for fld in dataset.metadata.record_sets[0].fields:
    if 'mw' in fld.name.lower() or 'weight' in fld.name.lower():
        numeric_field_id = fld.id  # assign molecular weight field's @id
    elif 'sample' in fld.name.lower() or 'condition' in fld.name.lower():
        group_field_id = fld.id  # assign sample or condition for grouping

# If field not found by guess, use printed field list above
if numeric_field_id is None:
    # Just use first float/int field found
    for fld in dataset.metadata.record_sets[0].fields:
        if fld.data_type in ["Float", "Integer", "Number"]:
            numeric_field_id = fld.id
            break
if group_field_id is None:
    # Use any other string field for demonstration
    for fld in dataset.metadata.record_sets[0].fields:
        if fld.data_type in ["Text", "String"]:
            group_field_id = fld.id
            break

if numeric_field_id not in df.columns:
    print(f"Warning: Numeric field {numeric_field_id} is not present in DataFrame.")
else:
    print(f"Filtering by field: {numeric_field_id} (threshold = 10)")
    threshold = 10
    df_filtered = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold}: {len(df_filtered)} found\n")
    print(df_filtered.head())

    # Normalize
    df_filtered[f"{numeric_field_id}_normalized"] = (
        df_filtered[numeric_field_id] - df_filtered[numeric_field_id].mean()
    ) / df_filtered[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    print(df_filtered[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by a group field if present
    if group_field_id and group_field_id in df_filtered.columns:
        grouped_df = df_filtered.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Update `numeric_field_id` and `group_field_id` as desired for other explorations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Grouped boxplot if group_field available
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45, ha='right')
        plt.show()

## 6. Conclusion

- We loaded the FAIR² dataset metadata using `mlcroissant` and explored protein-centric and/or sample-level record sets, referencing all entities by their `@id` fields as per the schema.
- We loaded all record sets into DataFrames, demonstrated simple exploratory analysis (filtering, normalization, grouping), and produced basic visualizations.

For further analysis, consult the dataset's Croissant schema for full field definitions and documentation.